# Groundwater flow (GWF) model

This notebook loads a pre-built MODFLOW 6 model of a synthetic valley, takes a look at its inputs, runs the groundwater flow (GWF) simulation, and plots the simulated heads. The companion notebook `flopy-intro-gwt-B.ipynb` reuses these flow results to run a groundwater transport (GWT) simulation.

## Imports and helper functions

Import FloPy and the plotting libraries used throughout the notebook. The `time_slider_view` helper draws a single interactive figure that updates as you move a time slider; it renders each frame to a PNG so exactly one figure is shown in both VS Code and JupyterLab.

In [ ]:
%matplotlib inline
import datetime
import io
import pathlib as pl

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import Image, IntSlider
from IPython.display import display
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

In [ ]:
# Drive a time-index slider that always shows exactly one figure, identically
# in VS Code and JupyterLab. Each frame is rendered to a PNG and pushed into a
# single ipywidgets.Image; assigning Image.value *replaces* the displayed frame
# in every frontend, so figures never stack. `make_fig(time_index)` must build
# and return a Matplotlib figure.
def time_slider_view(make_fig, ntimes, value=None, description="time index", dpi=200):
    image = Image(format="png")
    slider = IntSlider(
        min=0,
        max=ntimes - 1,
        step=1,
        value=ntimes - 1 if value is None else value,
        description=description,
        continuous_update=False,
    )

    def update(*_):
        fig = make_fig(slider.value)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi)
        plt.close(fig)  # keep the inline backend from also emitting the figure
        image.value = buf.getvalue()

    slider.observe(update, names="value")
    display(slider, image)
    update()

## Load the existing model

Read the existing simulation from the `data` directory and point it at a fresh `models/` workspace so the original input files are left untouched. We then grab the flow model and a few quantities (layer and cell counts, stress-period lengths, and the center of the lake) that the figures below rely on.

In [ ]:
# load the simple model setup

name = "sv"
ws = pl.Path("../data/synthetic-valley/synthetic-valley-vg")
new_ws = pl.Path(f"models/{ws.name}")

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)

In [ ]:
gwf = sim.get_model(name)
nlay, ncpl = gwf.disv.nlay.array, gwf.disv.ncpl.array
shape3d = (nlay, ncpl)

nper = sim.tdis.nper.array
perlen = sim.tdis.perioddata.array["perlen"]

# x-coordinate of the center of the lake (LAK) package; the model cross
# sections (A-A') are taken along this north-south line so they cut through
# the middle of the lake instead of the center of the model domain
lak_cellids = gwf.lak.connectiondata.array["cellid"]
lak_icpl = np.array(sorted({c[-1] for c in lak_cellids}))
lake_center_x = float(gwf.modelgrid.xcellcenters[lak_icpl].mean())

## Inspect the model inputs

Before running anything, plot the main inputs to get a feel for the model: aquifer properties, geometry, and the stresses applied over time.

### Horizontal hydraulic conductivity

Hydraulic conductivity (K) controls how easily water moves through each layer. Higher K (warmer colors) means more permeable material.

In [ ]:
# shared settings for the gwf map figures
gwf_figsize = (17.15 / 2.541, 1.4 * 0.8333 * 17.15 / 2.541)
gwf_mosaic = [[0, 1, 2], [3, 4, "."]]

idomain = gwf.disv.idomain.array

# horizontal hydraulic conductivity by layer
k11 = np.ma.masked_where(idomain < 1, gwf.npf.k.array)
kpos = k11.compressed()
norm_k = mpl.colors.LogNorm(vmin=kpos[kpos > 0].min(), vmax=kpos.max())

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        gwf_mosaic,
        figsize=gwf_figsize,
        layout="constrained",
    )
    fig.suptitle("Horizontal hydraulic conductivity (m/d)")

    for k in range(nlay):
        ax = axd[k]
        ax.set_xlim(gwf.modelgrid.extent[:2])
        ax.set_ylim(gwf.modelgrid.extent[2:])
        ax.set_title(f"Layer {k + 1}")
        mm = flopy.plot.PlotMapView(
            model=gwf,
            ax=ax,
            extent=gwf.modelgrid.extent,
            layer=k,
        )
        mm.plot_ibound(color_vpt="blue", alpha=0.25)
        pa = mm.plot_array(k11, norm=norm_k, edgecolor="black", lw=0.25)
        mm.plot_bc(package=gwf.sfr, color="cyan")
        mm.plot_bc(package=gwf.wel, color="red")

    fig.colorbar(
        pa,
        ax=[axd[k] for k in range(nlay)],
        shrink=0.6,
        label="K (m/d)",
    )

### Cell bottom elevation

The bottom elevation of every cell, layer by layer. This defines the vertical geometry (and thickness) of the model.

In [ ]:
# cell bottom elevation by layer
botm = np.ma.masked_where(idomain < 1, gwf.disv.botm.array)

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        gwf_mosaic,
        figsize=gwf_figsize,
        layout="constrained",
    )
    fig.suptitle("Cell bottom elevation (m)")

    for k in range(nlay):
        ax = axd[k]
        ax.set_xlim(gwf.modelgrid.extent[:2])
        ax.set_ylim(gwf.modelgrid.extent[2:])
        ax.set_title(f"Layer {k + 1}")
        mm = flopy.plot.PlotMapView(
            model=gwf,
            ax=ax,
            extent=gwf.modelgrid.extent,
            layer=k,
        )
        mm.plot_ibound(color_vpt="blue", alpha=0.25)
        pa = mm.plot_array(
            botm,
            vmin=botm.min(),
            vmax=botm.max(),
            cmap="terrain",
            edgecolor="black",
            lw=0.25,
        )
        mm.plot_bc(package=gwf.sfr, color="cyan")

    fig.colorbar(
        pa,
        ax=[axd[k] for k in range(nlay)],
        shrink=0.6,
        label="Elevation (m)",
    )

### Model top and cross-section location

The land-surface elevation across the model. The dashed A–A' line marks the north–south cross section (through the center of the lake) used in the section views below.

In [ ]:
# single figure of the model top
extent = gwf.modelgrid.extent
xext = extent[1] - extent[0]
yext = extent[3] - extent[2]
w = 17.15 / 2.541 / 2
single_figsize = (w, w * yext / xext)

top = np.ma.masked_where(idomain[0] < 1, gwf.disv.top.array)

# cross-section location through the center of the lake (matches the
# cross-section figures)
xc = lake_center_x

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=single_figsize, layout="constrained")
    ax.set_xlim(extent[:2])
    ax.set_ylim(extent[2:])
    ax.set_title("Model top (m)")
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, extent=extent, layer=0)
    mm.plot_ibound(color_vpt="blue", alpha=0.25)
    pa = mm.plot_array(top, cmap="terrain", edgecolor="black", lw=0.25)
    mm.plot_bc(package=gwf.sfr, color="cyan")

    # show the cross-section line
    ax.plot([xc, xc], extent[2:], color="red", lw=1.5, ls="--")
    ax.text(
        xc, extent[3], "A", color="red", fontweight="bold", ha="center", va="bottom"
    )
    ax.text(xc, extent[2], "A'", color="red", fontweight="bold", ha="center", va="top")

    fig.colorbar(pa, ax=ax, shrink=0.8, label="Elevation (m)")

### Hydraulic conductivity along section A–A'

The same hydraulic conductivity, now shown in cross section along A–A' so you can see how the layers stack vertically.

In [ ]:
# cross-section along section A-A' (center of the lake) showing horizontal K
extent = gwf.modelgrid.extent
xc = lake_center_x
# order the line A (north/top) -> A' (south/bottom) to match the map labels
line = {"line": [(xc, extent[3]), (xc, extent[2])]}

w = 17.15 / 2.541
with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(w, 0.5 * w), layout="constrained")
    ax.set_title("Horizontal hydraulic conductivity along section A-A'")
    xs = flopy.plot.PlotCrossSection(model=gwf, ax=ax, line=line)
    pa = xs.plot_array(gwf.npf.k.array, norm=norm_k)
    xs.plot_grid(lw=0.3, color="0.5")
    xs.plot_ibound()
    ax.set_xlabel("Distance along section (m)")
    ax.set_ylabel("Elevation (m)")

    # label the section endpoints to match the model-top figure
    ax.text(
        0.0,
        1.01,
        "A",
        color="red",
        fontweight="bold",
        ha="left",
        va="bottom",
        transform=ax.transAxes,
    )
    ax.text(
        1.0,
        1.01,
        "A'",
        color="red",
        fontweight="bold",
        ha="right",
        va="bottom",
        transform=ax.transAxes,
    )

    fig.colorbar(pa, ax=ax, shrink=0.8, label="K (m/d)")

### Applied recharge

Net recharge (rainfall reaching the water table) applied to the non-lake cells, plotted as a monthly depth over the transient simulation period.

In [ ]:
# recharge rate applied to non-lake cells, by stress period
# (the rcha array is spatially uniform within each stress period)
perlen = sim.tdis.perioddata.array["perlen"]
rch_arr = gwf.rcha.recharge.array  # (nper, 1, ncpl)
rch_rate = rch_arr.reshape(perlen.size, -1).mean(axis=1)  # m/d

# convert the daily rate (m/d) to a monthly depth (mm/month) using each
# stress period's length (days) and 1000 mm/m
rch_month = rch_rate * perlen * 1000.0

# period boundaries (days) relative to START_DATE_TIME = 1962-01-01
start_date = datetime.datetime(1962, 1, 1)
bounds = np.concatenate(([0.0], np.cumsum(perlen)))  # len nper + 1

# exclude the steady-state spin-up (period 1); keep the monthly transient
monthly = np.arange(1, perlen.size)
x_days = np.append(bounds[monthly], bounds[-1])  # period starts + final end time
y = np.append(rch_month[monthly], rch_month[monthly][-1])
dates = [start_date + datetime.timedelta(days=float(d)) for d in x_days]

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(
        figsize=(17.15 / 2.541, 0.45 * 17.15 / 2.541),
        layout="constrained",
    )
    ax.plot(dates, y, drawstyle="steps-post", color="blue", lw=1.0)
    ax.fill_between(dates, y, step="post", color="blue", alpha=0.2)
    ax.set_xlim(dates[0], dates[-1])
    ax.set_ylim(bottom=0.0)
    flopy.plot.styles.heading(
        ax=ax, heading="Applied net recharge during the transient simulation period"
    )
    flopy.plot.styles.xlabel(ax=ax, label="Date")
    flopy.plot.styles.ylabel(ax=ax, label="Recharge rate (mm/month)")

## Run the flow model

Write the input files and run the flow simulation, but only if the output is missing or incomplete. If the head and budget files already contain every saved time step, the run is skipped so re-running the notebook stays fast.

In [ ]:
# only run the gwf model if its output is missing or incomplete: the head
# and budget files must exist and contain every time step that the gwf OC
# package is configured to save


def gwf_oc_expected_kstpkper(gwf, sim):
    """(kstp, kper) pairs (zero-based) the gwf OC is set to save, per record type."""
    nstp = sim.tdis.perioddata.array["nstp"]
    saverecord = gwf.oc.saverecord.get_data()  # {kper: recarray} or None

    expected = {}  # rtype -> set of (kstp, kper)
    current = {}  # rtype -> (ocsetting, ocsetting_data); forward-filled over periods
    for kper in range(nstp.size):
        rec = saverecord.get(kper) if saverecord else None
        if rec is not None:
            for rtype, setting, data in rec:
                current[rtype.lower()] = (setting.lower(), data)
        ns = int(nstp[kper])
        for rtype, (setting, data) in current.items():
            if setting == "all":
                ksteps = range(ns)
            elif setting == "first":
                ksteps = [0]
            elif setting == "last":
                ksteps = [ns - 1]
            elif setting == "frequency":
                freq = int(data[0])
                ksteps = [k for k in range(ns) if (k + 1) % freq == 0]
            elif setting == "steps":
                want = {int(s) for s in data}
                ksteps = [k for k in range(ns) if (k + 1) in want]
            else:
                ksteps = range(ns)
            expected.setdefault(rtype, set()).update((k, kper) for k in ksteps)
    return expected


def gwf_output_available(gwf, sim, ws):
    """True only if every output file the gwf OC writes exists and is complete."""
    expected = gwf_oc_expected_kstpkper(gwf, sim)
    readers = {
        "head": (gwf.oc.head_filerecord.array, gwf.output.head),
        "budget": (gwf.oc.budget_filerecord.array, gwf.output.budget),
    }
    for rtype, want in expected.items():
        if not want:
            continue
        filerecord, reader = readers.get(rtype, (None, None))
        if reader is None or filerecord is None:
            return False
        # the output file must exist on disk
        if not (ws / filerecord[0][0]).is_file():
            return False
        # ...and contain every requested time step
        try:
            have = {(int(k), int(p)) for k, p in reader().get_kstpkper()}
        except Exception:
            return False
        if not want.issubset(have):
            return False
    return True


if gwf_output_available(gwf, sim, new_ws):
    print("gwf output is already complete; skipping sim.run_simulation()")
else:
    sim.write_simulation()
    sim.run_simulation()

## View the simulated heads

Plot the simulated groundwater levels (heads). Use the time slider to step through the simulation.

### Head maps through time

Simulated head in each layer at the selected time. Contours show equal-head lines, and the A–A' line marks the cross section shown below.

In [ ]:

# simulated heads through time (six-panel map; one frame per output time)
head_times = gwf.output.head().get_times()

extent = gwf.modelgrid.extent
xc = lake_center_x  # cross-section through the center of the lake (A-A')

head_mosaic = [[0, 1, 2], [3, 4, "cbar"]]
head_vmin, head_vmax = -2, 4
head_levels = np.arange(-2, 5, 1)
head_ticks = list(range(-2, 5))


def plot_head(time_index):
    totim = head_times[time_index]
    head = gwf.output.head().get_data(totim=totim)

    with flopy.plot.styles.USGSMap():
        fig, axd = plt.subplot_mosaic(
            head_mosaic,
            figsize=gwf_figsize,
            layout="constrained",
        )
        fig.suptitle(f"Simulated head (m), time = {totim:g} days (index {time_index})")

        for k in range(nlay):
            ax = axd[k]
            ax.set_xlim(extent[:2])
            ax.set_ylim(extent[2:])
            ax.set_title(f"Layer {k + 1}")
            mm = flopy.plot.PlotMapView(
                model=gwf,
                ax=ax,
                extent=extent,
                layer=k,
            )
            mm.plot_ibound(color_vpt="blue", alpha=0.25)
            pa = mm.plot_array(
                head,
                vmin=head_vmin,
                vmax=head_vmax,
                edgecolor="black",
                lw=0.25,
                masked_values=[1e30],
            )
            mm.plot_bc(package=gwf.sfr, color="cyan")
            mm.plot_bc(package=gwf.wel, color="red")

            cs = mm.contour_array(
                head,
                levels=head_levels,
                colors="black",
                linewidths=0.5,
                masked_values=[1e30],
            )
            ax.clabel(
                cs,
                inline=True,
                fmt="%g",
                fontsize=8,
                inline_spacing=0.5,
            )

            # show the cross-section location on the Layer 1 panel
            if k == 0:
                ax.plot([xc, xc], extent[2:], color="red", lw=1.5, ls="--")
                ax.text(
                    xc, extent[3], "A", color="red", fontweight="bold",
                    ha="center", va="bottom",
                )
                ax.text(
                    xc, extent[2], "A'", color="red", fontweight="bold",
                    ha="center", va="top",
                )

        # whole-number colorbar in the empty mosaic panel
        axd["cbar"].axis("off")
        cax = inset_axes(axd["cbar"], width="25%", height="85%", loc="center")
        cb = fig.colorbar(pa, cax=cax, ticks=head_ticks)
        cb.ax.yaxis.set_major_formatter(
            mpl.ticker.FuncFormatter(lambda x, _: f"{x:g}")
        )
        cb.set_label("Head (m)")

    return fig


time_slider_view(plot_head, len(head_times))

### Head along section A–A' through time

Simulated head in cross section along A–A', so you can watch the water table rise and fall through time.

In [ ]:
# simulated heads along section A-A' (center of the lake) through time
line = {"line": [(xc, extent[3]), (xc, extent[2])]}  # A (top) -> A' (bottom)
xs_w = 17.15 / 2.541


def plot_head_xs(time_index):
    totim = head_times[time_index]
    head = gwf.output.head().get_data(totim=totim)

    with flopy.plot.styles.USGSMap():
        fig, ax = plt.subplots(figsize=(xs_w, 0.5 * xs_w), layout="constrained")
        ax.set_title(
            f"Simulated head along A-A', time = {totim:g} days (index {time_index})"
        )
        xs = flopy.plot.PlotCrossSection(model=gwf, ax=ax, line=line)
        pa = xs.plot_array(
            head, head=head, vmin=head_vmin, vmax=head_vmax, masked_values=[1e30],
        )
        xs.plot_grid(lw=0.3, color="0.5")
        xs.plot_ibound()

        cs = xs.contour_array(
            head,
            head=head,
            levels=head_levels,
            colors="black",
            linewidths=0.5,
            masked_values=[1e30],
        )
        ax.clabel(cs, inline=True, fmt="%g", fontsize=8, inline_spacing=0.5)

        ax.set_xlabel("Distance along section (m)")
        ax.set_ylabel("Elevation (m)")

        # label the section endpoints to match the map figure
        ax.text(
            0.0, 1.01, "A", color="red", fontweight="bold",
            ha="left", va="bottom", transform=ax.transAxes,
        )
        ax.text(
            1.0, 1.01, "A'", color="red", fontweight="bold",
            ha="right", va="bottom", transform=ax.transAxes,
        )

        cb = fig.colorbar(pa, ax=ax, shrink=0.8, ticks=head_ticks)
        cb.ax.yaxis.set_major_formatter(
            mpl.ticker.FuncFormatter(lambda x, _: f"{x:g}")
        )
        cb.set_label("Head (m)")

    return fig


time_slider_view(plot_head_xs, len(head_times))